This notebook accomplishes two things:
- process HLR attributes for NWM CONUS, AK, and HI domains 
- contruct the initital attributes configuration file (that can later be used to select attributes for regionalization)

In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq

In [2]:
# define data directories
ngen_reg_dir = Path("/home/yuqiong.liu/work/data/ngen_reg/")

In [3]:
# function to get info about attribute datasets
def get_attr_info(attr: str) -> dict:
    attr_info = dict()
    match attr.lower():
        case "hlr":
            attr_info["file"] = "/home/yuqiong.liu/work/data/HLR/hlrshape/hlrus.shp"
            attr_info["id_column"] = "VALUE"
            attr_info["layer"] = "hlrus"
            attr_info["attr_list"] = [
                "AQPERMNEW",
                "SLOPE",
                "TAVE",
                "PPT",
                "PET",
                "SAND",
                "PMPE",
                "MINELE",
                "RELIEF",
                "PFLATTOT",
                "PFLATLOW",
                "PFLATUP",
            ]
        case "hydroatlas":
            attr_info["file"] = (
                "/home/yuqiong.liu/work/data/HydroATLAS/BasinATLAS_v10_shp/BasinATLAS_v10_lev12.shp"
            )
            attr_info["id_column"] = "PFAF_ID"
            attr_info["layer"] = "BasinATLAS_v10_lev12"
            attr_info["attr_list"] = []
        case _:
            raise Exception(f"Unsupported attribute dataset: {attr}")

    return attr_info

In [4]:
attr_dataset = "hlr"
attr_dict = get_attr_info(attr_dataset)
subs_all = gpd.read_file(attr_dict["file"], layer=attr_dict["layer"])
df_attrs = subs_all[[attr_dict["id_column"]] + attr_dict["attr_list"]].copy()

In [5]:
# loop through domains to process attributes
for domain in ["conus", "ak", "hi"]:
    print(domain)

    # ngen-hlr crosswalk file
    cwt_file = Path(
        ngen_reg_dir,
        "cwt_ngen_"
        + attr_dataset
        + "/cwt_ngen_"
        + attr_dataset
        + "_"
        + domain
        + ".parquet",
    )

    # peek into the cwt file
    # pf = pq.ParquetFile(cwt_file)
    # print("Number of row groups:", pf.num_row_groups)
    # print("Number of rows:", pf.metadata.num_rows)
    # print("Schema:")
    # print(pf.schema)

    # read the crosswalk file
    df_cwt = pd.read_parquet(cwt_file)

    # remove rows where no overlapping subbasin were found (hence the nearest subbasins were identified instead; not used here)
    df_cwt = df_cwt[df_cwt["nearest_dist_m"].isna()]

    # merge attributes dataset with crosswalk
    df_attrs1 = df_attrs.merge(df_cwt, on=attr_dict["id_column"], how="outer")
    df_attrs1 = df_attrs1[(df_attrs1["VALUE"] > 0) & (~df_attrs1["divide_id"].isna())]
    df_attrs1 = df_attrs1[
        ["divide_id"] + [col for col in df_attrs1.columns if col != "divide_id"]
    ]

    # Multiply values by weights
    cols1 = attr_dict["attr_list"]
    weighted = df_attrs1[cols1].multiply(df_attrs1["overlap_percentage"], axis=0)
    weighted["divide_id"] = df_attrs1["divide_id"]

    # Group by and sum
    weighted_sum = weighted.groupby("divide_id").sum()

    # Divide by sum of weights per group to get weighted mean
    sum_weights = df_attrs1.groupby("divide_id")["overlap_percentage"].sum()
    weighted_mean = weighted_sum[cols1].div(sum_weights, axis=0).reset_index()

    # save attr data to parquet file
    outfile = Path(
        ngen_reg_dir, "attr_datasets/attr_" + attr_dataset + "_" + domain + ".parquet"
    )
    weighted_mean.to_parquet(outfile, engine="pyarrow")

conus
ak
hi


In [6]:
# Prepare the attributes config file for the HLR dataset
df_attr_config = pd.DataFrame(
    columns=["select", "attr_name", "description"],
    data=[
        (1, "AQPERMNEW", "aquifer permeability"),
        (0, "SLOPE", "mean slope"),
        (1, "TAVE", "mean annual temperature"),
        (1, "PPT", "mean annual precipitation"),
        (1, "PET", "mean annual potential evaporatranspiration"),
        (1, "PMPE", "mean annual precpitation minus PET"),
        (1, "SAND", "percentage of sand in the soil"),
        (0, "MINELE", "minimum elevation"),
        (0, "RELIEF", "relief"),
        (0, "PFLATTOT", "total percentage of flatland"),
        (0, "PFLATLOW", "percentage of flatlad in the lowland area"),
        (0, "PFALTUP", "percentage of flatlat in the upland area"),
    ],
)

df_attr_config.to_csv(
    Path(ngen_reg_dir, "config/attr_selection_" + attr_dataset + ".csv"),
    index=False,
    header=True,
)

In [7]:
print(df_attr_config)

    select  attr_name                                 description
0        1  AQPERMNEW                        aquifer permeability
1        0      SLOPE                                  mean slope
2        1       TAVE                     mean annual temperature
3        1        PPT                   mean annual precipitation
4        1        PET  mean annual potential evaporatranspiration
5        1       PMPE          mean annual precpitation minus PET
6        1       SAND              percentage of sand in the soil
7        0     MINELE                           minimum elevation
8        0     RELIEF                                      relief
9        0   PFLATTOT                total percentage of flatland
10       0   PFLATLOW   percentage of flatlad in the lowland area
11       0    PFALTUP    percentage of flatlat in the upland area
